# CGPA Prediction

Two models that predict `Final_CGPA` from study hours, attendance, sleep, social hours and previous CGPA.

The table is `data/Student_data.csv` (5,000 rows). The split is 4,000 / 1,000, `random_state=42`, which is the same split the original project used.

- Multiple linear regression: `sklearn.linear_model.LinearRegression` (ordinary least squares, intercept on, no scaling).
- Random forest: `sklearn.ensemble.RandomForestRegressor` with 100 trees and `random_state=42`.

Run all cells from the repository root, or from `notebooks/` — the paths below walk up one directory. The command-line version of this walkthrough is `src/predict_cgpa.py`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

ROOT = Path("..") if Path("../data/Student_data.csv").is_file() else Path(".")
df = pd.read_csv(ROOT / "data" / "Student_data.csv")
print(df.shape)
print("missing cells:", int(df.isna().sum().sum()))
df.head()

## Summary statistics

Mean, median (the 50% row), quartiles, and the first mode of each numeric column. Gender and major are collected in the table but are not inputs to either model.


In [ ]:
numeric = [
    "Age", "Attendance_Pct", "Study_Hours_Per_Day", "Previous_CGPA",
    "Sleep_Hours", "Social_Hours_Week", "Final_CGPA",
]
print(df[numeric].describe().T)
print("\nModes:")
print(df[numeric].mode().iloc[0])
print("\nGender:")
print(df["Gender"].value_counts())
print("\nMajor:")
print(df["Major"].value_counts())

## Correlations and outliers

Pearson correlations of the numeric columns, then Tukey IQR outlier counts (1.5 × IQR). I leave the outliers in, the same as the original notebook.


In [ ]:
print(df[numeric].corr().round(4))
print()
for col in numeric:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = int(((df[col] < lo) | (df[col] > hi)).sum())
    print(f"{col:22s}  outliers={n:3d}  fence=({lo:.3f}, {hi:.3f})")

## Fit both models

Five predictors, `Final_CGPA` as the target, 80/20 split, `random_state=42`.


In [ ]:
features = [
    "Study_Hours_Per_Day",
    "Attendance_Pct",
    "Sleep_Hours",
    "Social_Hours_Week",
    "Previous_CGPA",
]
X = df[features]
y = df["Final_CGPA"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("train", len(X_train), "test", len(X_test))

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

def report(name, pred):
    mse = mean_squared_error(y_test, pred)
    print(f"{name}")
    print(f"  MSE  {mse:.12f}")
    print(f"  RMSE {np.sqrt(mse):.8f}")
    print(f"  MAE  {mean_absolute_error(y_test, pred):.8f}")
    print(f"  R2   {r2_score(y_test, pred):.12f}")

report("Linear Regression (test)", lr_pred)
report("Random Forest (test)", rf_pred)

print("\nCoefficients:")
print(f"  intercept  {lr.intercept_:.12f}")
for name, coef in zip(features, lr.coef_):
    print(f"  {name:22s} {coef:.12f}")

print("\nRandom-forest importances:")
for name, val in zip(features, rf.feature_importances_):
    print(f"  {name:22s} {val:.12f}")

## Learned equation

Six decimal places, matching `src/predict_cgpa.py`.


In [ ]:
terms = [f"{lr.intercept_:.6f}"]
for name, coef in zip(features, lr.coef_):
    sign = "+" if coef >= 0 else "-"
    terms.append(f"{sign} {abs(coef):.6f}*{name}")
print("Final_CGPA = " + " ".join(terms))

## First ten test predictions


In [ ]:
pred = pd.DataFrame({
    "Student_ID": df.loc[X_test.index, "Student_ID"].values,
    "Actual": y_test.values,
    "Pred_LR": lr_pred,
    "Pred_RF": rf_pred,
    "Err_LR": lr_pred - y_test.values,
    "Err_RF": rf_pred - y_test.values,
})
pred.head(10)

## Charts

Actual vs predicted, and a residual plot for the linear model. The command-line script writes a fuller set under `results/figures/`.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, pred_vals, title in [
    (axes[0], lr_pred, "Linear regression"),
    (axes[1], rf_pred, "Random forest"),
]:
    ax.scatter(y_test, pred_vals, s=12, alpha=0.4)
    lo = min(y_test.min(), pred_vals.min())
    hi = max(y_test.max(), pred_vals.max())
    ax.plot([lo, hi], [lo, hi], color="black", linewidth=1)
    ax.set_xlabel("Actual Final CGPA")
    ax.set_ylabel("Predicted Final CGPA")
    ax.set_title(title)
fig.tight_layout()
plt.show()

resid = y_test.to_numpy() - lr_pred
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(lr_pred, resid, s=12, alpha=0.4)
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("Predicted Final CGPA")
ax.set_ylabel("Residual (actual − predicted)")
ax.set_title("Residuals — linear regression")
fig.tight_layout()
plt.show()